# DBSCAN Algorithm - Educational Implementation

A bare-minimum implementation of DBSCAN (Density-Based Spatial Clustering of Applications with Noise) for learning purposes.

## Algorithm Overview

1. **Parameter Selection**: Choose ε (epsilon) and MinPts
2. **Select Starting Point**: Pick an arbitrary unvisited point
3. **Examine Neighborhood**: Find all points within ε distance
4. **Expand Cluster**: Recursively add neighbors if core point
5. **Repeat**: Process all unvisited points
6. **Finalize**: Classify points as core, border, or noise

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

## Core DBSCAN Implementation

In [ ]:
class DBSCAN:
    def __init__(self, epsilon, min_pts):
        """
        Initialize DBSCAN with parameters.
        
        Parameters:
        -----------
        epsilon : float
            Maximum distance between two points to be neighbors
        min_pts : int
            Minimum number of points to form a dense region
        """
        self.epsilon = epsilon
        self.min_pts = min_pts
        self.labels = None
        
    def _euclidean_distance(self, point1, point2):
        """Calculate Euclidean distance between two points."""
        return np.sqrt(np.sum((point1 - point2) ** 2))
    
    def _get_neighbors(self, data, point_idx):
        """
        Step 3: Examine the Neighborhood
        Find all points within epsilon distance of the given point.
        
        Returns:
        --------
        List of indices of neighboring points
        """
        neighbors = []
        for idx in range(len(data)):
            if self._euclidean_distance(data[point_idx], data[idx]) <= self.epsilon:
                neighbors.append(idx)
        return neighbors
    
    def _expand_cluster(self, data, point_idx, neighbors, cluster_id, visited):
        """
        Step 4: Expand the Cluster
        Add all reachable points to the current cluster.
        
        Parameters:
        -----------
        data : array
            Dataset
        point_idx : int
            Index of the core point
        neighbors : list
            Indices of neighboring points
        cluster_id : int
            Current cluster ID
        visited : set
            Set of visited point indices
        """
        # Assign the core point to the cluster
        self.labels[point_idx] = cluster_id
        
        # Use a queue for breadth-first expansion
        queue = deque(neighbors)
        
        while queue:
            current_point = queue.popleft()
            
            # If point was noise, it's now a border point
            if self.labels[current_point] == -1:
                self.labels[current_point] = cluster_id
            
            # Skip if already processed
            if current_point in visited:
                continue
                
            visited.add(current_point)
            self.labels[current_point] = cluster_id
            
            # Find neighbors of current point
            current_neighbors = self._get_neighbors(data, current_point)
            
            # If it's a core point, add its neighbors to the queue
            if len(current_neighbors) >= self.min_pts:
                for neighbor in current_neighbors:
                    if neighbor not in visited:
                        queue.append(neighbor)
    
    def fit(self, data):
        """
        Apply DBSCAN algorithm to the dataset.
        
        Parameters:
        -----------
        data : array-like, shape (n_samples, n_features)
            Dataset to cluster
            
        Returns:
        --------
        labels : array, shape (n_samples,)
            Cluster labels (-1 for noise, 0+ for cluster IDs)
        """
        data = np.array(data)
        n_points = len(data)
        
        # Initialize all points as noise (-1)
        self.labels = np.full(n_points, -1)
        
        # Track visited points
        visited = set()
        
        # Current cluster ID
        cluster_id = 0
        
        # Step 2 & 5: Select starting point and repeat for all points
        for point_idx in range(n_points):
            # Skip if already visited
            if point_idx in visited:
                continue
            
            visited.add(point_idx)
            
            # Step 3: Examine the neighborhood
            neighbors = self._get_neighbors(data, point_idx)
            
            # If not enough neighbors, mark as noise (for now)
            if len(neighbors) < self.min_pts:
                self.labels[point_idx] = -1
            else:
                # It's a core point - expand the cluster
                self._expand_cluster(data, point_idx, neighbors, cluster_id, visited)
                cluster_id += 1
        
        return self.labels

## Generate Sample Data

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Create three clusters with different centers
cluster1 = np.random.randn(50, 2) * 0.5 + np.array([0, 0])
cluster2 = np.random.randn(50, 2) * 0.5 + np.array([5, 5])
cluster3 = np.random.randn(50, 2) * 0.5 + np.array([0, 5])

# Add some noise points
noise = np.random.uniform(-2, 7, (20, 2))

# Combine all data
data = np.vstack([cluster1, cluster2, cluster3, noise])

print(f"Generated dataset with {len(data)} points")
print(f"Data shape: {data.shape}")

## Step 1: Parameter Selection

In [ ]:
# Set DBSCAN parameters
epsilon = 0.5      # Maximum distance for neighborhood
min_pts = 5        # Minimum points to form dense region

print(f"Parameters:")
print(f"  ε (epsilon): {epsilon}")
print(f"  MinPts: {min_pts}")

## Run DBSCAN Algorithm

In [ ]:
# Create and fit DBSCAN
dbscan = DBSCAN(epsilon=epsilon, min_pts=min_pts)
labels = dbscan.fit(data)

# Count clusters and noise points
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"\nResults:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points: {n_noise}")
print(f"  Cluster labels: {set(labels)}")

## Visualize Results

In [ ]:
# Create visualization
plt.figure(figsize=(12, 5))

# Plot original data
plt.subplot(1, 2, 1)
plt.scatter(data[:, 0], data[:, 1], alpha=0.6, s=50)
plt.title('Original Data')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True, alpha=0.3)

# Plot clustered data
plt.subplot(1, 2, 2)
unique_labels = set(labels)
colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        # Noise points in black
        color = 'black'
        marker = 'x'
        label_name = 'Noise'
    else:
        marker = 'o'
        label_name = f'Cluster {label}'
    
    mask = labels == label
    plt.scatter(data[mask, 0], data[mask, 1], 
               c=[color], label=label_name, 
               alpha=0.6, s=50, marker=marker)

plt.title(f'DBSCAN Results (ε={epsilon}, MinPts={min_pts})')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Experiment with Different Parameters

In [ ]:
# Try different parameter combinations
param_combinations = [
    (0.3, 5),
    (0.5, 5),
    (0.8, 5),
    (0.5, 3),
    (0.5, 10)
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (eps, min_p) in enumerate(param_combinations):
    dbscan_temp = DBSCAN(epsilon=eps, min_pts=min_p)
    labels_temp = dbscan_temp.fit(data)
    
    n_clusters = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
    n_noise = list(labels_temp).count(-1)
    
    # Plot
    unique_labels = set(labels_temp)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            color = 'black'
            marker = 'x'
        else:
            marker = 'o'
        
        mask = labels_temp == label
        axes[idx].scatter(data[mask, 0], data[mask, 1], 
                         c=[color], alpha=0.6, s=30, marker=marker)
    
    axes[idx].set_title(f'ε={eps}, MinPts={min_p}\n'
                       f'Clusters: {n_clusters}, Noise: {n_noise}')
    axes[idx].grid(True, alpha=0.3)

# Hide the last subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()

## Summary

This implementation demonstrates the core DBSCAN algorithm:

1. **Parameter Selection**: ε and MinPts control cluster density
2. **Neighbor Search**: Points within ε distance are neighbors
3. **Core Points**: Points with ≥ MinPts neighbors
4. **Cluster Expansion**: Recursively add reachable points
5. **Border Points**: Non-core points in a cluster
6. **Noise**: Points not belonging to any cluster

### Key Observations:
- Smaller ε creates more, tighter clusters
- Larger ε merges clusters together
- Higher MinPts requires denser regions
- DBSCAN can find arbitrarily shaped clusters
- No need to specify number of clusters beforehand